In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import numpy as np

class TradingVisualizer:
    def __init__(self, env):
        self.env = env
        self.current_step = env.lookback_window
        
    def render_step(self, step=None):
        """Render a specific step"""
        if step is None:
            step = self.current_step
        else:
            self.current_step = step
            
        # Get data for current lookback window
        start_idx = max(0, step - self.env.lookback_window)
        end_idx = step + 1  # +1 to include current step
        window_data = self.env.data.iloc[start_idx:end_idx]
        
        # Get current state from history
        current_state = self.env.history[step]
        
        # Create the chart
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.7, 0.3],
            shared_yaxes=True,
            horizontal_spacing=0.02,
            subplot_titles=('Price Action', 'Volume Profile')
        )
        
        # 1. Candlestick chart (left)
        fig.add_trace(
            go.Candlestick(
                x=window_data.index,
                open=window_data['open'],
                high=window_data['high'],
                low=window_data['low'],
                close=window_data['close'],
                name="Price"
            ),
            row=1, col=1
        )
        
        # Add current price line
        current_price = self.env.data['close'].iloc[step]
        fig.add_hline(y=current_price, line_dash="dash", line_color="red", 
                     annotation_text=f"Price: {current_price:.2f}", row=1, col=1)
        
        # Add position markers if any
        if current_state['position_type'] != 0:
            entry_price = current_state['entry_price']
            position_type = "LONG" if current_state['position_type'] == 1 else "SHORT"
            color = "green" if position_type == "LONG" else "red"
            fig.add_hline(y=entry_price, line_dash="dot", line_color=color,
                         annotation_text=f"Entry ({position_type})", row=1, col=1)
        
        # 2. Volume Profile (right)
        current_vp = self.env.volume_profiles[step]
        
        # Get price range from current window for bin centers
        price_min = window_data['low'].min()
        price_max = window_data['high'].max() 
        bin_centers = np.linspace(price_min, price_max, len(current_vp))
        
        fig.add_trace(
            go.Bar(
                x=current_vp,
                y=bin_centers,
                orientation='h',
                name="Volume Profile",
                marker_color='rgba(100, 100, 200, 0.6)'
            ),
            row=1, col=2
        )
        
        # Add current price to volume profile
        fig.add_hline(y=current_price, line_dash="dash", line_color="red", row=1, col=2)
        
        # Update layout
        action_map = {0: "FLAT", 1: "LONG", 2: "SHORT"}
        current_action = action_map.get(current_state.get('action', 0), "FLAT")
        
        fig.update_layout(
            title=f"Step {step} | Equity: ${current_state['equity']:.2f} | Action: {current_action} | Position: {current_state['position_type']}",
            height=600,
            showlegend=False,
            xaxis_rangeslider_visible=False
        )
        
        fig.update_xaxes(title_text="Time", row=1, col=1)
        fig.update_xaxes(title_text="Volume Density", row=1, col=2)
        fig.update_yaxes(title_text="Price", row=1, col=1)
        
        return fig.show()
    
    def create_interactive_controller(self):
        """Create slider and buttons to navigate through steps"""
        
        max_step = len(self.env.history) - 1
        
        # Create widgets
        step_slider = widgets.IntSlider(
            value=self.current_step,
            min=self.env.lookback_window,
            max=max_step,
            step=1,
            description='Step:',
            continuous_update=False,
            layout=widgets.Layout(width='80%')
        )
        
        prev_btn = widgets.Button(description='◀ Previous', layout=widgets.Layout(width='100px'))
        next_btn = widgets.Button(description='Next ▶', layout=widgets.Layout(width='100px'))
        play_btn = widgets.Play(
            interval=500,
            value=self.current_step,
            min=self.env.lookback_window,
            max=max_step,
            step=1
        )
        
        step_label = widgets.Label(value=f"Step: {self.current_step}/{max_step}")
        
        # Output display
        output = widgets.Output()
        
        def update_display(step):
            self.current_step = step
            step_slider.value = step
            step_label.value = f"Step: {step}/{max_step}"
            with output:
                output.clear_output(wait=True)
                self.render_step(step)
        
        def on_slider_change(change):
            update_display(change['new'])
        
        def on_prev_click(btn):
            new_step = max(self.env.lookback_window, self.current_step - 1)
            update_display(new_step)
        
        def on_next_click(btn):
            new_step = min(max_step, self.current_step + 1)
            update_display(new_step)
        
        def on_play_change(change):
            update_display(change['new'])
        
        # Wire up events
        step_slider.observe(on_slider_change, names='value')
        prev_btn.on_click(on_prev_click)
        next_btn.on_click(on_next_click) 
        play_btn.observe(on_play_change, names='value')
        
        # Layout
        controls = widgets.HBox([prev_btn, play_btn, next_btn])
        slider_box = widgets.VBox([step_slider, controls, step_label])
        
        # Initial display
        update_display(self.current_step)
        
        display(widgets.VBox([slider_box, output]))

# Usage:
def add_visualizer_to_env(env):
    """Add visualizer to your existing environment"""
    visualizer = TradingVisualizer(env)
    
    # Add methods to env
    env.visualizer = visualizer
    env.render_step = visualizer.render_step
    env.create_controller = visualizer.create_interactive_controller
    
    return visualizer

In [10]:

# Configuration
import os

from environments.simple_trading_env import SimpleTradingEnv
from training.data_loader import DataLoader


data_dir = os.path.join(os.getcwd(), "data_simple")
#datasets = ['BTCUSDT-1.json', 'BTCUSDT-2.json', 'BTCUSDT-3.json', 'BTCUSDT-4.json', 'BTCUSDT-5.json']

def create_env_for_dataset(data):
    """Create environment for specific dataset"""
    return SimpleTradingEnv(
        data=data,
        initial_balance=10000,
        lookback_window=50,  # Adjust based on your data
        # Add other env parameters you need
    )


# Train individual models with SPARSE REWARD optimization
models = []
training_logs = []


# Load data
data = DataLoader.load_json_data_file(f"{data_dir}\\BTCUSDT-1.json")
dataset_size = len(data)
print(f"Loaded {dataset_size} data points.")

# Create environment
env = create_env_for_dataset(data)

# After running your environment
visualizer = add_visualizer_to_env(env)

# Method 3: Use the visualizer directly
visualizer.render_step(3)
visualizer.create_interactive_controller()

Loaded 1371 data points.


KeyError: 'position_type'